# Phase 3 — Reasoning finetuning, evaluation and attention comparison (Kaggle)

One committed run does **everything** for Phase 3: finetunes Model H (Telugu) and
Model L (Nepali) from their own pretrained checkpoints, evaluates pretrained vs
finetuned on the synthetic reasoning test sets, checks language-modelling
retention, runs the pretrained-vs-finetuned attention comparison, and packages
`report/phase3/` for download. Budget: **~30–45 min on one T4** — each finetune
is ~0.26M tokens/epoch × 30 epochs (6,000 examples).

**Attach three data sources** (right panel → Add Data):

| Dataset | What the notebook needs from it | Discovered by |
|---|---|---|
| `lma-code` (**new version** — Phase 3 code) | `common/`, `telugu/`, `nepali/`, `scripts/`, `tests/` | `common/finetune.py` |
| Phase 2 checkpoints (your notebook output or dataset) | `telugu/best.pt`, `nepali/best.pt` | `**/telugu/best.pt` |
| `lma-tokenized` (optional) | `<lang>/tokenized/{validation,test}.bin` + `token_counts.json` for the forgetting check | `telugu/tokenized/test.bin` |

Then **Save Version → Save & Run All (Commit)**. Settings → Accelerator → GPU T4.

## 1 — Environment

In [ ]:
import torch, os
from pathlib import Path

print("torch     :", torch.__version__)
print("CUDA      :", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings > Accelerator > GPU T4, then rerun.")
print("device    :", torch.cuda.get_device_name(0))
print("VRAM      :", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

print("\n/kaggle/input tree (3 levels):")
root = Path("/kaggle/input")
if not any(root.iterdir()):
    print("   EMPTY - no datasets attached.")
for a in sorted(root.iterdir()):
    print("  ", a.name + "/")
    if a.is_dir():
        for b in sorted(a.iterdir())[:8]:
            print("      ", b.name + ("/" if b.is_dir() else ""))
            if b.is_dir():
                for c in sorted(b.iterdir())[:6]:
                    print("          ", c.name + ("/" if c.is_dir() else ""))

## 2 — Discover inputs and configure

Everything is found by searching for a file that must exist, so Kaggle's mount
slugs never need to be typed. If a checkpoint is found in more than one place
(e.g. two notebook versions), the candidates are printed and the **first** is
used — check it is the one you mean.

Set `RESTORE_*` only when continuing an interrupted finetune from a previous
version's output (`/kaggle/input/<prev-slug>/checkpoints/telugu_reasoning`).

In [ ]:
import os
from pathlib import Path

RESTORE_TE = None      # e.g. "/kaggle/input/<prev-notebook-slug>/checkpoints/telugu_reasoning"
RESTORE_NE = None
RUN_TESTS  = True      # ~2 min; catches a stale code upload before training

def find(marker, label, required=True):
    hits = sorted(Path("/kaggle/input").glob(f"**/{marker}"))
    if not hits:
        if required:
            raise SystemExit(f"'{marker}' not found under /kaggle/input -- the {label} dataset is not attached.")
        print(f"  note: {label} not attached ({marker}); that stage will be skipped")
        return None
    if len(hits) > 1:
        print(f"  note: {len(hits)} candidates for {label}:")
        for h in hits: print("        ", h)
        print("        using the first")
    return hits[0]

def root_of(hit, marker):
    p = hit
    for _ in range(len(Path(marker).parts)): p = p.parent
    return str(p)

def find_ckpt(lang):
    # Accepts <dir>/<lang>/best.pt, <lang>_best.pt, best_<lang>.pt, <lang>*best*.pt.
    # Never picks the ablation ('nopos').
    pats = [f"**/{lang}/best.pt", f"**/{lang}_best.pt", f"**/best_{lang}.pt",
            f"**/{lang}*best*.pt", f"**/*{lang}*/best.pt"]
    hits = []
    for pat in pats:
        hits += [h for h in sorted(Path("/kaggle/input").glob(pat))
                 if "nopos" not in str(h) and h not in hits]
    if not hits:
        raise SystemExit(f"no pretrained best.pt for {lang} under /kaggle/input "
                         f"(tried {pats}). Check the lma-ckpts dataset layout in cell 1.")
    if len(hits) > 1:
        print(f"  note: {len(hits)} candidates for {lang} checkpoint: {hits}; using the first")
    return str(hits[0])

CODE    = root_of(find("common/finetune.py", "code"), "common/finetune.py")
CKPT_TE = find_ckpt("telugu")
CKPT_NE = find_ckpt("nepali")
data_hit = find("telugu/tokenized/test.bin", "tokenized corpora", required=False)
DATA    = root_of(data_hit, "telugu/tokenized/test.bin") if data_hit else None

os.chdir("/kaggle/working")
print(f"  CODE    : {CODE}")
print(f"  CKPT_TE : {CKPT_TE}")
print(f"  CKPT_NE : {CKPT_NE}")
print(f"  DATA    : {DATA or '(none - LM forgetting check will be skipped)'}")

## 3 — Stage code, regenerate the reasoning datasets, write Kaggle configs

The datasets are deterministic (seed 1337 / 2718) and regenerate in seconds;
`tests/test_reasoning.py::test_committed_dataset_matches_generator` proves the
regenerated test split equals the committed one. Configs get Kaggle paths
written into `finetune_kaggle.yaml`; the committed `finetune.yaml` is untouched.

In [ ]:
import shutil, os, yaml, subprocess, sys
from pathlib import Path

for item in ("common", "telugu", "nepali", "scripts", "tests"):
    dst = f"/kaggle/working/{item}"
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(f"{CODE}/{item}", dst)
for d in ("report/phase3/logs", "report/phase3/stats", "report/phase3/figures", "checkpoints"):
    os.makedirs(d, exist_ok=True)

checks = {
    "finetune.py --session-steps": "session_steps" in Path("common/finetune.py").read_text(),
    "reasoning_eval.py":           Path("common/reasoning_eval.py").exists(),
    "attention_compare.py":        "def piece_offsets" in Path("common/attention_compare.py").read_text(),
    "run_phase3.py --pretrained-root": "--pretrained-root" in Path("scripts/run_phase3.py").read_text(),
    "telugu lexicon":              Path("telugu/reasoning/lexicon.py").exists(),
    "nepali lexicon":              Path("nepali/reasoning/lexicon.py").exists(),
}
for k, v in checks.items(): print(f"  {'OK  ' if v else 'FAIL'}  {k}")
assert all(checks.values()), "code dataset is stale - upload a new version of lma-code"

print()
for lang in ("telugu", "nepali"):
    r = subprocess.run([sys.executable, "-m", f"{lang}.reasoning.generate"], capture_output=True, text=True)
    print(r.stdout.strip().splitlines()[-3:] if r.returncode == 0 else r.stderr[-2000:])
    assert r.returncode == 0

CKPT = {"telugu": CKPT_TE, "nepali": CKPT_NE}
for lang in ("telugu", "nepali"):
    c = yaml.safe_load(Path(f"{lang}/configs/finetune.yaml").read_text(encoding="utf-8"))
    c["init_from"] = CKPT[lang]
    if DATA:
        c["paths"]["train_bin"] = f"{DATA}/{lang}/tokenized/train.bin"
        c["paths"]["val_bin"]   = f"{DATA}/{lang}/tokenized/validation.bin"
        c["paths"]["test_bin"]  = f"{DATA}/{lang}/tokenized/test.bin"
    Path(f"{lang}/configs/finetune_kaggle.yaml").write_text(
        yaml.safe_dump(c, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print(f"  {lang}: init_from={c['init_from']}")

## 4 — Tests

In [ ]:
if RUN_TESTS:
    !python tests/test_reasoning.py 2>&1 | grep -v INFO | tail -20
else:
    print("skipped")

## 5 — Finetune Model H (Telugu)

`--resume` is always safe: with nothing to resume from it starts fresh from the
pretrained checkpoint. The log goes to `report/phase3/logs/telugu_finetune.jsonl`.

In [ ]:
import shlex
restore = f"--restore-from {shlex.quote(RESTORE_TE)}" if RESTORE_TE else ""
!python -m common.finetune --config telugu/configs/finetune_kaggle.yaml --resume {restore} 2>&1 | grep -v "^$"

## 6 — Finetune Model L (Nepali)

In [ ]:
restore = f"--restore-from {shlex.quote(RESTORE_NE)}" if RESTORE_NE else ""
!python -m common.finetune --config nepali/configs/finetune_kaggle.yaml --resume {restore} 2>&1 | grep -v "^$"

## 7 — Evaluate: reasoning (pretrained vs finetuned), LM retention, attention comparison

`run_phase3.py` reads the finetuned checkpoints from `checkpoints/<lang>_reasoning/`
and the pretrained ones from the read-only input. Every stage caches its JSON, so
re-running after an interruption skips what is done.

In [ ]:
import shutil, os
# The orchestrator expects <pretrained-root>/<lang>/best.pt.
os.makedirs("pretrained/telugu", exist_ok=True); os.makedirs("pretrained/nepali", exist_ok=True)
for lang, src in (("telugu", CKPT_TE), ("nepali", CKPT_NE)):
    dst = f"pretrained/{lang}/best.pt"
    if not os.path.exists(dst):
        os.symlink(src, dst)          # no 310MB copy into the output

# The orchestrator reads <lang>/configs/finetune.yaml; point those at the Kaggle configs.
for lang in ("telugu", "nepali"):
    shutil.copy(f"{lang}/configs/finetune_kaggle.yaml", f"{lang}/configs/finetune.yaml")

!python scripts/run_phase3.py --langs telugu nepali --pretrained-root pretrained 2>&1 | grep -v "^$"

## 8 — Results at a glance

In [ ]:
import json, glob
from pathlib import Path
for lang in ("telugu", "nepali"):
    print(f"== {lang}")
    for tag in ("pretrained", "finetuned"):
        p = Path(f"report/phase3/stats/{lang}_{tag}_reasoning_test.json")
        if p.exists():
            r = json.loads(p.read_text(encoding="utf-8"))
            print(f"   {tag:10s} exact match {r['exact_match']:.4f} | choice {r['choice_accuracy']:.4f} (chance {r['choice_chance']:.4f})")
            print("              by task:", {k: v['accuracy'] for k, v in r['exact_match_by_task'].items()})
    log = Path(f"report/phase3/logs/{lang}_finetune.jsonl")
    if log.exists():
        rows = [json.loads(l) for l in log.read_text(encoding="utf-8").splitlines() if l.strip()]
        ev = [r for r in rows if "val_exact_match" in r]
        fin = [r for r in rows if "final_val_exact_match" in r]
        if ev: print(f"   val EM curve: {[e['val_exact_match'] for e in ev]}")
        if fin: print(f"   final: {fin[-1]}")
print()
cmp = Path("report/phase3/stats/comparison.md")
print(cmp.read_text(encoding="utf-8")[:3000] if cmp.exists() else "(comparison.md not written -- check cell 7 for errors)")

## 9 — Package for download

`phase3_report.zip` holds everything that goes into the repo (`report/phase3/`:
logs, stats, figures). The finetuned checkpoints (`checkpoints/*_reasoning/best.pt`
and `latest.pt`) go to Google Drive; put the links in the README.

In [ ]:
import shutil, os, glob
# restore the committed configs so the output tree matches the repo
for lang in ("telugu", "nepali"):
    shutil.copy(f"{CODE}/{lang}/configs/finetune.yaml", f"{lang}/configs/finetune.yaml")
shutil.make_archive("/kaggle/working/phase3_report", "zip", "/kaggle/working", "report/phase3")
print("report zip :", os.path.getsize("/kaggle/working/phase3_report.zip")/1e6, "MB")
!ls -la /kaggle/working/checkpoints/*/
print("\nUpload to Drive:")
for p in sorted(glob.glob("/kaggle/working/checkpoints/*_reasoning/*.pt")):
    print("  ", p, f"{os.path.getsize(p)/1e6:.0f} MB")